In [42]:
import pandas as pd
import numpy as np

In [43]:
import os
os.chdir("D:/churn-mlops")
data=pd.read_csv('data/churn.csv')

In [44]:
data['gender'].unique()
data=data.drop(['customerID'],axis=1)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
SeniorCitizen,1.000000,0.016567,0.220173,0.102411
tenure,0.016567,1.000000,0.247900,0.825880
MonthlyCharges,0.220173,0.247900,1.000000,0.651065
TotalCharges,0.102411,0.825880,0.651065,1.000000


In [45]:
col=data.columns

In [46]:
col

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [47]:
for i in col:
    print(i,data[i].unique())

gender ['Female' 'Male']
SeniorCitizen [0 1]
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
tenure [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService ['No' 'Yes']
MultipleLines ['No phone service' 'No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
MonthlyCharges [29.85 56.95 53.85 ... 63.1  44.2  78.7 ]
TotalCharges ['29.85' '1889

In [48]:
target=data['Churn']

In [49]:
data=data.drop(['Churn'],axis=1)

In [50]:
target = target.apply(lambda x: 1 if x == "Yes" else 0)

In [51]:
data['tenure'].corr(target)

np.float64(-0.3522286701130777)

In [52]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
x = scaler.fit_transform(data[["tenure"]])

In [53]:
x

array([[0.01388889],
       [0.47222222],
       [0.02777778],
       ...,
       [0.15277778],
       [0.05555556],
       [0.91666667]], shape=(7043, 1))

In [87]:
target


0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [88]:
type(target)

pandas.core.series.Series

In [56]:
type(x)

numpy.ndarray

In [57]:
xn.corr(target)

np.float64(-0.35222867011307796)

In [58]:
data['MonthlyCharges'].corr(target)

np.float64(0.19335642223784694)

In [59]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

In [60]:
data["TotalCharges"].corr(target)

np.float64(-0.1994840835675645)

In [61]:
data[data['TotalCharges'].isna()]['tenure'].describe()

count    11.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: tenure, dtype: float64

In [69]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import joblib

def encode_categorical(df):
    df = df.copy()

    # 1. Fix TotalCharges (blank strings -> numeric)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0) #filling zero as the null values correspond to non - customers

    # 2. Binary columns -> LabelEncoder (2 classes only)
    binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    label_encoders = {}
    for col in binary_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le  # save each one, you'll need them at inference time

    # 3. Multi-category columns -> OneHotEncoder
    multi_cat_cols = [
        'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
        'Contract', 'PaymentMethod'
    ]
    ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoded_array = ohe.fit_transform(df[multi_cat_cols])
    encoded_df = pd.DataFrame(
        encoded_array,
        columns=ohe.get_feature_names_out(multi_cat_cols),
        index=df.index
    )

    df = df.drop(columns=multi_cat_cols)
    df = pd.concat([df, encoded_df], axis=1)

    # 4. Save the encoders — you'll load these in your FastAPI app later
    
    joblib.dump(label_encoders, "D:/churn-mlops/models/label_encoders.pkl")
    joblib.dump(ohe, "D:/churn-mlops/models/onehot_encoder.pkl")

    return df


In [71]:
df = encode_categorical(data)
print(df.shape)
df.to_csv("data/churn_processed.csv", index=False)

(7043, 30)


In [73]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,0,56.95,1889.50,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,1,0,0,0,2,1,1,53.85,108.15,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,45,0,0,42.30,1840.75,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0,0,0,0,2,1,1,70.70,151.65,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,1,84.80,1990.50,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
7039,0,0,1,1,72,1,1,103.20,7362.90,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
7040,0,0,1,1,11,0,1,29.60,346.45,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7041,1,1,1,0,4,1,1,74.40,306.60,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [89]:
t = target.to_frame(name="Target")

In [90]:
t

,Target
0,0
1,0
2,1
3,0
4,1
...,...
7038,0
7039,0
7040,0
7041,1


In [91]:
a=pd.concat([df,target],axis=1)

In [92]:
a

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
0,0,0,1,0,1,0,1,29.85,29.85,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,1,0,0,0,34,1,0,56.95,1889.50,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0
2,1,0,0,0,2,1,1,53.85,108.15,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
3,1,0,0,0,45,0,0,42.30,1840.75,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
4,0,0,0,0,2,1,1,70.70,151.65,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,1,84.80,1990.50,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0
7039,0,0,1,1,72,1,1,103.20,7362.90,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0
7040,0,0,1,1,11,0,1,29.60,346.45,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
7041,1,1,1,0,4,1,1,74.40,306.60,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1


In [94]:
a=a.corr()

In [100]:
a['Churn'].sort_values(ascending=False)

Churn                                    1.000000
InternetService_Fiber optic              0.308020
PaymentMethod_Electronic check           0.301919
MonthlyCharges                           0.193356
PaperlessBilling                         0.191825
SeniorCitizen                            0.150889
StreamingTV_Yes                          0.063228
StreamingMovies_Yes                      0.061382
MultipleLines_Yes                        0.040102
PhoneService                             0.011942
gender                                  -0.008612
MultipleLines_No phone service          -0.011942
DeviceProtection_Yes                    -0.066160
OnlineBackup_Yes                        -0.082255
PaymentMethod_Mailed check              -0.091683
PaymentMethod_Credit card (automatic)   -0.134302
Partner                                 -0.150448
Dependents                              -0.164221
TechSupport_Yes                         -0.164674
OnlineSecurity_Yes                      -0.171226


In [108]:
#Executing Logistic Regression with 5 fold cross val
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Example: assume df is your DataFrame and 'Churn' is the target column
X = df   # features
y = target               # target

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# Evaluate with cross-validation (AUC or accuracy)
scores = cross_val_score(model, X_scaled, y, cv=kf, scoring="roc_auc")

print("Cross-validation AUC scores:", scores)
print("Mean AUC:", scores.mean())


Cross-validation AUC scores: [0.86211274 0.86374637 0.82968739 0.83345188 0.84230679]
Mean AUC: 0.8462610326570035


In [112]:

import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
# Example: assume df is your DataFrame and 'Churn' is the target column
X = df   # features
y = target               # target

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print(df_results)
print("\nMean metrics across folds:")
print(df_results.mean())

        AUC  Accuracy        F1  Precision    Recall
0  0.862113  0.821150  0.638968   0.686154  0.597855
1  0.863746  0.809794  0.620397   0.663636  0.582447
2  0.829687  0.801987  0.601997   0.674121  0.543814
3  0.833452  0.794034  0.579710   0.636943  0.531915
4  0.842307  0.810369  0.583463   0.656140  0.525281

Mean metrics across folds:
AUC          0.846261
Accuracy     0.807467
F1           0.604907
Precision    0.663399
Recall       0.556262
dtype: float64


In [120]:
#Executing Logistic Regression with increased number of folds in k fold cross val
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
# Example: assume df is your DataFrame and 'Churn' is the target column
X = df   # features
y = target               # target

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=50, shuffle=True, random_state=42)

results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print(df_results)
print("\nMean metrics across folds:")
print(df_results.mean())

         AUC  Accuracy        F1  Precision    Recall
0   0.900656  0.851064  0.734177   0.828571  0.659091
1   0.884324  0.843972  0.620690   0.600000  0.642857
2   0.845309  0.765957  0.620690   0.658537  0.586957
3   0.882540  0.829787  0.657143   0.676471  0.638889
4   0.844802  0.794326  0.567164   0.703704  0.475000
5   0.833962  0.851064  0.695652   0.705882  0.685714
6   0.842593  0.836879  0.610169   0.692308  0.545455
7   0.873786  0.794326  0.641975   0.604651  0.684211
8   0.877249  0.836879  0.610169   0.782609  0.500000
9   0.844075  0.794326  0.567164   0.633333  0.513514
10  0.857545  0.787234  0.625000   0.694444  0.568182
11  0.835344  0.773050  0.542857   0.612903  0.487179
12  0.845118  0.808511  0.630137   0.741935  0.547619
13  0.869272  0.801418  0.588235   0.606061  0.571429
14  0.837805  0.780142  0.597403   0.638889  0.560976
15  0.884392  0.843972  0.685714   0.705882  0.666667
16  0.864286  0.822695  0.666667   0.641026  0.694444
17  0.890128  0.801418  0.64

In [115]:
#Executing Logistic Regression with decreased number of folds in k fold cross val
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
# Example: assume df is your DataFrame and 'Churn' is the target column
X = df   # features
y = target               # target

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=3, shuffle=True, random_state=42)


results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print(df_results)
print("\nMean metrics across folds:")
print(df_results.mean())

        AUC  Accuracy        F1  Precision    Recall
0  0.860862  0.816440  0.634436   0.686239  0.589905
1  0.840072  0.802811  0.600518   0.669231  0.544601
2  0.835455  0.799318  0.573756   0.622790  0.531879

Mean metrics across folds:
AUC          0.845463
Accuracy     0.806190
F1           0.602903
Precision    0.659420
Recall       0.555462
dtype: float64


In [116]:
#Trying Logistic Regression with removing the cols with less correlation to the target
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
# Example: assume df is your DataFrame and 'Churn' is the target column
X = df.drop([
    
        "StreamingTV_Yes",
        "StreamingMovies_Yes",
        "MultipleLines_Yes",
        "PhoneService",
        "gender",
        "MultipleLines_No phone service",
        "DeviceProtection_Yes",
        "OnlineBackup_Yes",
        "PaymentMethod_Mailed check",
    
],axis=1)   # features
y = target               # target

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)


results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print(df_results)
print("\nMean metrics across folds:")
print(df_results.mean())

        AUC  Accuracy        F1  Precision    Recall
0  0.862520  0.814053  0.613569   0.681967  0.557641
1  0.858708  0.799148  0.593974   0.644860  0.550532
2  0.825053  0.801278  0.597701   0.675325  0.536082
3  0.830224  0.787642  0.569784   0.620690  0.526596
4  0.841133  0.804688  0.574961   0.639175  0.522472

Mean metrics across folds:
AUC          0.843528
Accuracy     0.801362
F1           0.589998
Precision    0.652403
Recall       0.538665
dtype: float64


In [117]:
target.describe()

count    7043.000000
mean        0.265370
std         0.441561
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: Churn, dtype: float64

In [118]:
target.info()

<class 'pandas.core.series.Series'>
RangeIndex: 7043 entries, 0 to 7042
Series name: Churn
Non-Null Count  Dtype
--------------  -----
7043 non-null   int64
dtypes: int64(1)
memory usage: 55.2 KB


In [119]:
#Discovering imbalance in the dataset
count=target.value_counts()
count

Churn
0    5174
1    1869
Name: count, dtype: int64

In [122]:
#Adjusting class weights so as to achieve more accuracy
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

# Example: assume df is your churn DataFrame
X = df   # features
y = target              # target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Logistic regression with class weights to handle imbalance
model = LogisticRegression(
    max_iter=1000,
    solver="liblinear",
    class_weight="balanced"   # <-- key change
)

# Define k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Collect metrics for each fold
results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print("Metrics per fold:\n", df_results)
print("\nMean metrics across folds:\n", df_results.mean())


Metrics per fold:
         AUC  Accuracy        F1  Precision    Recall
0  0.861994  0.748048  0.633643   0.515101  0.823056
1  0.863427  0.759404  0.654434   0.530579  0.853723
2  0.831129  0.733144  0.616327   0.510135  0.778351
3  0.832715  0.744318  0.621053   0.513937  0.784574
4  0.841231  0.741477  0.599119   0.492754  0.764045

Mean metrics across folds:
 AUC          0.846099
Accuracy     0.745278
F1           0.624915
Precision    0.512501
Recall       0.800750
dtype: float64


In [125]:
#Trying resampling to combat data imbalance in target
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE

# Example: assume df is your churn DataFrame
X = df   # features
y = target               # target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Logistic regression model
model = LogisticRegression(max_iter=1000, solver="liblinear")

# Define k-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Collect metrics for each fold
results = []
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Apply SMOTE only on the training set
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    # Train model
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    # Collect metrics
    metrics = {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }
    results.append(metrics)

# Convert to DataFrame for comparison
df_results = pd.DataFrame(results)
print("Metrics per fold:\n", df_results)
print("\nMean metrics across folds:\n", df_results.mean())


Metrics per fold:
         AUC  Accuracy        F1  Precision    Recall
0  0.862767  0.756565  0.645295   0.525253  0.836461
1  0.863888  0.765082  0.660513   0.537563  0.856383
2  0.829145  0.734564  0.612836   0.512111  0.762887
3  0.833521  0.750710  0.630137   0.521815  0.795213
4  0.841167  0.749290  0.600227   0.502846  0.744382

Mean metrics across folds:
 AUC          0.846098
Accuracy     0.751242
F1           0.629801
Precision    0.519917
Recall       0.799065
dtype: float64


In [124]:
pip install imblearn


   ---------------------------------------- 0/3 [sklearn-compat]
   ---------------------------------------- 0/3 [sklearn-compat]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ---------------------------------------- 3/3 [imblearn]

Note:


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
